# DynaBench — validation, batch evaluation, and evidence rules
**Research pilot v3.2 · Colab GPU · existing Python 3.10 environment**

This extends the working cleaned notebook. Its baseline remains the **same current-code YOLOE + SigLIP path**,
not the paper's verified OWL-based setup. No upstream source, model, color path, or baseline thresholds are changed.

**Already completed:** one supplied real `toy owl` prediction, geometric error **27.51 cm** within a **30 cm** tolerance.
It was much closer to the green-toy annotation, so semantic identity still needs the box audit below.
**Completed pilot:** both scenes were run, outputs were audited, and the qualified conclusions are in `docs/RESULTS.md`. The notebook deliberately clears saved cell outputs for readability; rerunning it requires a GPU runtime and downloads. The age-plus-similarity result was **not a robust demonstrated improvement**.

### Run order
Run the numbered sections in order in a **hosted GPU runtime**. The bootstrap skips an exactly matching working environment and repairs recorded-core version drift.
The notebook embeds its source/tests, so **you do not need to upload a separate source ZIP**. Old data and diagnostic
folders are preserved. Every model command uses `/content/stretch310/bin/python`, never Colab's default interpreter.

The default split is **env1 for development, env2 for held-out evaluation**. env1 has **54 primary questions**.
A separately labelled inclusive-cutoff sensitivity branch runs those 54 queries again, because official checkpoint
semantics are still unverified. It is never pooled with, or selected instead of, the primary branch.

**Guardrails:** unknown labels are not scored as negatives; exceptions are not declines; policy settings freeze before
held-out inference; one test scene supports only an exploratory pilot. No paid API calls are used.

**v3.1 fixes only setup:** the final compatibility install can no longer silently replace
the recorded torch/torchvision pair. Research code and experiment settings are unchanged.

**v3.2 input-schema repair:** Section 9 now validates a separate four-stream input copy for the held-out scene. The study modules and code hashes, environment bootstrap, model configuration, completed predictions, source data, and frozen policies are unchanged. In the current runtime, replace/run **only Section 9**, then continue with Sections 10–11. Do not rerun Sections 1–8 just to apply this repair.

## 1. Load or safely update the research code
Updates only recognized v3 setup/documentation files, with backups. Unknown local edits stop the update.
The model and experiment modules are unchanged; existing data, outputs and frozen study plans are preserved.
The embedded bundle means no separate source ZIP upload is required.

In [ ]:
from pathlib import Path
import base64, hashlib, io, zipfile

DEST = Path("/content/DynaBench_Research_v3")  # Preserve existing project/module paths.
PAYLOAD = """
...
"""
EXPECTED_SHA256 = '287c86455aa05b91d474aac0c4c00e1c4f0a042dea233afec61c09ea78ed88aa'
UPGRADE_FROM_V3 = {'README.md': '33ff9fcd366f64be27e19fe5abc2219a830b403cedb37ce6246d2e9f14838af6', 'VALIDATION.md': '92fe6ec44c33e162f72caac7dab26b80bd9eec7a62f2ee49622e36b98e65cae0', 'bootstrap_colab.sh': 'a7d4a0ae1bed2ed771bb56d4ab83bbae13395e22ce13d931ac4d66935351055e'}
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256, "Embedded source integrity failure"
DEST.mkdir(parents=True, exist_ok=True)

# Validate the complete update before writing anything. Unknown local edits are preserved.
pending = []
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        path = DEST / member.filename
        if not path.resolve().is_relative_to(DEST.resolve()):
            raise RuntimeError("Unsafe embedded path")
        content = archive.read(member.filename)
        backup = None
        if path.exists() and path.read_bytes() != content:
            old = path.read_bytes()
            allowed = UPGRADE_FROM_V3.get(member.filename)
            if allowed is None or hashlib.sha256(old).hexdigest() != allowed:
                raise RuntimeError(f"Preserve your edited file before loading this version: {path}")
            backup = path.with_name(path.name + ".before_v3_1.bak")
            if backup.exists() and backup.read_bytes() != old:
                raise RuntimeError(f"A different backup already exists; preserve it: {backup}")
        pending.append((path, content, backup))

for path, content, backup in pending:
    path.parent.mkdir(parents=True, exist_ok=True)
    if backup is not None and not backup.exists():
        backup.write_bytes(path.read_bytes())
    if not path.exists() or path.read_bytes() != content:
        temp = path.with_name(path.name + ".v3_1.tmp")
        temp.write_bytes(content)
        temp.replace(path)
print("Research code ready (v3.1 setup correction):", DEST)
print("Source archive SHA256:", EXPECTED_SHA256)
print("Experiment modules, reference settings, datasets and prior results are unchanged.")


## 2. Reuse, repair, or reconstruct the environment
**Exact recorded core + working GPU:** no package installation or upgrade.
**Version drift from the old bootstrap:** restore the recorded CUDA 12.6 torch pair in place,
then apply the known compatibility versions without resolving dependencies again.
**Fresh runtime:** install the stack with GPU constraints on every resolving install.

The script records before/after package inventories and verifies exact versions plus a GPU calculation.
It does not delete the dataset, results, project, virtual environment, or study plan.
The two known dependency-metadata conflicts remain explicit; this is not a clean full-stack lockfile.
Section 3 still performs the full model-import/SigLIP preflight before any replay.


In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
bash /content/DynaBench_Research_v3/bootstrap_colab.sh

## 3. Check the harness and freeze the study plan
The tests use **synthetic fixtures and mock models**, not benchmark results. The plan locks the scene split,
primary cutoff, feature choices, and seed. The preflight verifies the recorded working core package versions,
CUDA, pinned source, and SigLIP text encoding. A mismatch stops rather than silently changing the experiment.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory

cd /content/DynaBench_Research_v3
"$PY" -m unittest discover -s tests -v > /content/dynabench_v3_tests.txt 2>&1 || { cat /content/dynabench_v3_tests.txt; exit 1; }
tail -5 /content/dynabench_v3_tests.txt
"$PY" -m vmresearch.cli init --base "$BASE"
"$PY" -m vmresearch.cli preflight --base "$BASE"

## 4. Reuse env1 and audit the checkpoint convention
Downloads only missing files from the pinned dataset revision. Creates a **label-free** query manifest and checks
RGB/depth/camera alignment. Source matches and boundary images are saved for review.

**Important:** a source grep, 42 frames, or a good score cannot prove what checkpoint `14` means.
The notebook distinguishes the verified one-based internal observation IDs from the **unverified dataset cutoff**.
It preserves exclusive frames `0–13` as the primary assumption and tests the alternative separately.
Fully blank coordinates remain unknown unless their meaning is independently established.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
"$PY" -m vmresearch.cli prepare --base "$BASE" --scene env1

In [ ]:
from pathlib import Path
import json
base = Path("/content/visual_memory")
audit = json.loads((base / "prepared/env1/convention_audit.json").read_text())
print("Checkpoint status:", audit["status"])
print("Internal ID mapping:", audit["observation_id_mapping"])
print("Dataset cutoff:", audit["dataset_checkpoint_mapping"])
print("Potential author-source excerpts:", len(audit["source_evidence_candidates"]))
for item in audit["source_evidence_candidates"][:8]:
    print(f"\n{item['path']}:{item['line']}\n{item['excerpt']}")
print("Boundary images and the full evidence scan are saved in prepared/env1/.")

## 5. Audit the detector's choice for the supplied owl result
Replays **only its saved supporting frame (array index 12)** through the same YOLOE coordinate function.
A read-only callback records the box selected by that exact call; it does not choose a new box or rerun a different detector.

It checks whether the replayed coordinate matches the saved prediction, saves the box/crop and actual detector input,
and only afterward projects the owl/green-toy annotation anchors. This helps identify the selected object; the geometric
anchors are not pixel-level identity ground truth. Optional human review is stored separately and never becomes a policy feature.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
"$PY" -m vmresearch.cli audit-prior --base "$BASE"

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image
folder = Path("/content/visual_memory/prior_audit")
r = json.loads((folder / "detector_replay.json").read_text())
print("Matches the prior saved coordinate:", r["matches_saved_location_at_1e_6m"])
print("Selected box:", r["detector_trace"].get("selected_box_pixels"))
print("Detector confidence:", r["detector_trace"].get("selected_confidence"))
for name in ("selected_box.png", "projected_label_anchors.png"):
    print(name)
    display(Image(filename=str(folder / name)))
print("The actual detector-input color image and optional manual_review.json are saved alongside these images.")

## 6. Run all env1 questions, then score
Processes the observation stream once. Saves **54 exclusive-cutoff predictions** and the separate **54 inclusive sensitivity predictions**.
Each record includes its supporting frame, age in observation steps, similarity, detector box/confidence, return route, and timing.
No custom rejection gate is active yet. Answers are saved before scoring labels are read.

Completed questions are reusable. Interrupted runs rebuild the seeded memory once and skip completed queries.
An error is saved as an **error**, never relabelled as a decline; policy fitting stops if any inference error remains.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory

"$PY" -m vmresearch.cli batch --base "$BASE" --scene env1
"$PY" -m vmresearch.cli score --base "$BASE" --scene env1

## 7. Inspect mistakes before freezing rules
The complete per-query image gallery is `scoring/env1/review.html`, included in the final ZIP.
The summary below flags wrong geometric locations, positive-query declines, and possible wrong-object matches.
These flags describe observations; age alone does not prove staleness, and proximity alone does not prove identity.

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image
base = Path("/content/visual_memory")
metrics = json.loads((base / "scoring/env1/metrics.json").read_text())
print("PRIMARY METRICS (unknown blank labels excluded from correctness):")
print(json.dumps(metrics["exclusive"], indent=2))
rows = [json.loads(line) for line in (base / "scoring/env1/scored.jsonl").read_text().splitlines()]
flags = [r for r in rows if r["cutoff"] == "exclusive" and
         (r["outcome"] in ("wrong_location", "decline", "inference_error") or r.get("possible_wrong_object_match"))]
print("\nFlagged development cases:", len(flags))
for r in flags[:6]:
    print({k: r.get(k) for k in ("query_id", "query", "outcome", "distance_error_m",
                                "age_observation_steps", "return_route", "nearest_annotation_query")})
for r in flags[:3]:
    image = base / "runs/env1/exclusive" / r["query_id"] / "selected_box.png"
    if image.exists():
        print(r["query_id"], r["query"])
        display(Image(filename=str(image)))

## 8. Freeze the age-only and combined-evidence rules
Uses **env1 only**. The simple combined rule uses available, varying evidence: recency, query similarity,
and detector support. It is skipped when its required signals are missing. The weights are fixed equally,
not optimized on held-out results and not interpreted as calibrated probabilities.

Numeric thresholds target 25%, 50%, 75%, and 100% retention of development baseline answers.
Ties and resulting actual coverage are reported. The **75% point is predeclared as the primary comparison**.
The saved rules are immutable and hash-checked. Do not edit them after inspecting env2.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
"$PY" -m vmresearch.cli freeze --base "$BASE"

## 9. Prepare the held-out scene safely and run the same baseline
**In an existing session, this is the only cell to replace for the schema error.** No reinstall or development-scene replay is needed.

The original loader requires exactly `rgb`, `depth`, `camera_poses`, and `camera_K`.
That rejects additional top-level fields even when all four required streams are present.
The error log does not show the actual `env2` keys; this cell prints and checks them first.

The adapter:
- verifies the frozen policies and unchanged study code before reading held-out inputs;
- reuses the downloaded source files (downloads pinned env2 only if missing);
- writes a separate input copy containing only those four required streams, without changing array values, dtypes, frame order, units, geometry, or checkpoint files;
- retains the original loader's calibration/shape checks and compares per-stream fingerprints before and after serialization;
- records excluded keys and raw/derived file hashes, then runs the unchanged batch and scoring commands.

The raw data and completed `env1` outputs remain untouched. This is a derived input container, not a new raw-dataset revision.
No missing-key aliases or alternate layouts are guessed. Unsupported input prints an explicit diagnostic and stops before the GPU batch.
Extra fields are not assumed scientifically meaningless: the checkpoint convention remains unverified, and no cutoff or policy is changed.

The audit and adapter source are saved in `prepared/env2/` and included by the existing final exporter.
Resume uses the same verified input copy. Do not delete lock files or re-freeze policies to bypass a failure.


In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
FIX=/content/repair_env2_schema.py

test -x "$PY"
cat > /content/repair_env2_schema.py.incoming <<'PY_SCHEMA_ADAPTER'
"""Validate a non-destructive, four-stream scene view for the unchanged v3 harness.

This repair does not edit vmresearch, raw data, model settings, study locks,
preflight records, predictions, or frozen policies. It does not guess key aliases,
geometry, units, timing conventions, or the meaning of extra fields.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import shutil
import tempfile
from pathlib import Path

import numpy as np

from vmresearch.core import (REVISION, atomic_json, code_digest, init_project,
                             read_json, scene_role, sha256)
from vmresearch.data import ArrayTensorUnpickler, download_scene, load_scene, prepare_scene
from vmresearch.policies import load_frozen

REQUIRED = ("rgb", "depth", "camera_poses", "camera_K")
REPAIR_ID = "env2-four-stream-view-v1"


def fingerprint_streams(env: dict) -> dict:
    """Hash each frame's type, shape, dtype and numeric bytes in original order."""
    result = {}
    for key in REQUIRED:
        values = env[key]
        try:
            count = len(values)
        except TypeError as exc:
            raise ValueError(f"{key}: expected a sequence of frames") from exc
        h = hashlib.sha256()
        h.update(f"{type(values).__module__}.{type(values).__name__}:{count}".encode())
        for i, value in enumerate(values):
            a = np.asarray(value)
            if a.dtype.hasobject:
                raise ValueError(f"{key}[{i}]: object arrays are not supported; no conversion attempted")
            header = json.dumps({"index": i, "type": f"{type(value).__module__}.{type(value).__name__}",
                                 "shape": list(a.shape), "dtype": a.dtype.str}, sort_keys=True).encode()
            body = a.tobytes(order="C")
            h.update(len(header).to_bytes(8, "little")); h.update(header)
            h.update(len(body).to_bytes(8, "little")); h.update(body)
        result[key] = {"frames": count, "sha256": h.hexdigest()}
    return result


def protected_hashes(base: Path, protocol: dict) -> dict:
    """Read-only snapshots of the frozen plan and existing development results."""
    paths = [base / p for p in ("protocol.json", "plan_lock.json", "preflight.json",
                               "model_lock.json", "analysis/frozen_policies.json")]
    for scene in protocol["development_scenes"]:
        paths += [base / folder / scene / name for folder, name in (
            ("runs", "run_manifest.json"), ("runs", "predictions.jsonl"),
            ("runs", "completion.json"), ("scoring", "scored.jsonl"), ("scoring", "metrics.json"))]
    return {str(p.relative_to(base)): sha256(p) for p in paths if p.is_file()}


def stage_and_prepare(base: Path, repo: Path, scene: str = "env2", *, allow_download: bool = True) -> dict:
    base = base.resolve(); repo = repo.resolve()
    p = init_project(base)  # Keeps the existing protocol AND harness-hash guard.
    if scene_role(p, scene) != "evaluation":
        raise ValueError("This repair is only for a predeclared held-out scene, not development data")
    load_frozen(base)  # Verify rules and development-score hashes; never refit.
    before = protected_hashes(base, p)
    raw = base / "data" / "DynaBench" / scene
    if not (raw / "env.pkl").is_file() or not (raw / "time.txt").is_file():
        if not allow_download:
            raise FileNotFoundError(f"Scene not downloaded: {raw}")
        raw = download_scene(base, scene)
    with (raw / "env.pkl").open("rb") as f:
        env = ArrayTensorUnpickler(f).load()
    actual = [str(k) for k in env] if isinstance(env, dict) else None
    print("Raw scene type:", type(env).__name__, flush=True)
    print("Raw scene keys:", actual, flush=True)
    if not isinstance(env, dict):
        raise ValueError(f"Unsupported scene container {type(env).__name__}; no layout was guessed. "
                         "Share the Raw scene type/keys output. Original data and study are unchanged.")
    missing = [k for k in REQUIRED if k not in env]
    if missing:
        raise ValueError(f"Missing required fields: {missing}; available keys: {actual}. "
                         "This is not an extra-fields-only case. No renaming or geometry conversion was attempted.")
    extra = sorted(str(k) for k in env if k not in REQUIRED)
    view = {k: env[k] for k in REQUIRED}  # Same objects, no slicing/transposing/scaling/reordering.
    original_fingerprints = fingerprint_streams(view)
    print("Fields excluded from the model-input copy:", extra, flush=True)
    print("Original frame counts:", {k: v["frames"] for k, v in original_fingerprints.items()}, flush=True)
    # CSV files are copied byte-for-byte; their coordinate labels are not parsed here.
    source_files = [raw / "env.pkl", raw / "time.txt"] + sorted(
        f for f in raw.glob("*.csv") if f.stem.isdecimal())
    if len(source_files) <= 2:
        raise ValueError("No checkpoint CSV files found")
    raw_hashes = {f.name: sha256(f) for f in source_files}
    root = base / "data" / "DynaBench_model_inputs"
    dest = root / scene
    prepared = base / "prepared" / scene
    if (prepared / "manifest.json").exists():
        old_root = Path(read_json(prepared / "manifest.json")["data_root"]).resolve()
        if old_root != dest.resolve():
            raise RuntimeError("A scene manifest already points elsewhere. It was not overwritten; "
                               "review that existing prepared scene before changing its input path.")
    adapter_source = Path(__file__).resolve()
    adapter_hash = sha256(adapter_source)
    audit_name = "schema_adapter_audit.json"
    root.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        if not (dest / audit_name).is_file():
            raise RuntimeError(f"Unrecognized existing input-copy directory, left untouched: {dest}")
        audit = read_json(dest / audit_name)
        if (audit.get("repair_id") != REPAIR_ID or audit.get("source_files_sha256") != raw_hashes
                or audit.get("adapter_sha256") != adapter_hash):
            raise RuntimeError("Existing input copy has different source/adapter provenance; not overwritten")
        if audit.get("protected_before") != before:
            raise RuntimeError("Frozen plan or development result hashes changed since this copy was created")
        if sha256(dest / "env.pkl") != audit["model_input_pickle_sha256"]:
            raise RuntimeError("Model-input pickle changed after staging")
        normalized = load_scene(dest / "env.pkl")
        if fingerprint_streams(normalized) != original_fingerprints:
            raise RuntimeError("Input-copy stream values differ from the original scene")
        for f in source_files[1:]:
            if sha256(dest / f.name) != raw_hashes[f.name]:
                raise RuntimeError(f"Copied checkpoint file changed: {f.name}")
        if sha256(dest / "schema_adapter.py") != adapter_hash:
            raise RuntimeError("Saved adapter source changed")
        print("Reusing verified input copy:", dest, flush=True)
        del normalized
    else:
        import pickle
        # Commit the directory only after validation; a failed validation leaves no final copy.
        with tempfile.TemporaryDirectory(prefix=f".{scene}-staging-", dir=root) as temp:
            stage = Path(temp)
            with (stage / "env.pkl").open("wb") as f:
                pickle.dump(view, f, protocol=4)
            normalized = load_scene(stage / "env.pkl")  # Retain ALL original frame/camera checks.
            if fingerprint_streams(normalized) != original_fingerprints:
                raise RuntimeError("Serialization changed a required stream; aborting")
            del normalized
            for f in source_files[1:]:
                shutil.copyfile(f, stage / f.name)
                if sha256(stage / f.name) != raw_hashes[f.name]:
                    raise RuntimeError(f"Checkpoint-file copy verification failed: {f.name}")
            shutil.copyfile(adapter_source, stage / "schema_adapter.py")
            audit = {
                "repair_id": REPAIR_ID,
                "source_dataset_revision": REVISION,
                "original_scene_directory": str(raw.resolve()),
                "model_input_directory": str(dest.resolve()),
                "source_files_sha256": raw_hashes,
                "model_input_pickle_sha256": sha256(stage / "env.pkl"),
                "required_streams": list(REQUIRED), "original_top_level_keys": actual,
                "excluded_top_level_keys": extra, "stream_fingerprints": original_fingerprints,
                "adapter_sha256": adapter_hash, "unchanged_harness_sha256": code_digest(),
                "protected_before": before,
                "scope": "Four-stream projection only. No data values, frame order, units, geometry, "
                         "checkpoints, models, policies, labels, or scores were changed.",
                "limitations": "Excluded fields are not automatically meaningless. Timing/blank-label "
                               "conventions remain unresolved. No key aliases were guessed. "
                               "This is a derived input container, not an unmodified raw pickle.",
            }
            atomic_json(stage / audit_name, audit)
            stage.rename(dest)
    # All raw source files must still be unchanged.
    if {f.name: sha256(f) for f in source_files} != raw_hashes:
        raise RuntimeError("Raw source changed during staging")
    del view, env
    manifest = prepare_scene(base, scene, repo, data_root=root)
    # The existing exporter includes prepared/*, so the repair's provenance travels with results.
    shutil.copyfile(dest / audit_name, prepared / audit_name)
    shutil.copyfile(adapter_source, prepared / "schema_adapter.py")
    if protected_hashes(base, p) != before:
        raise RuntimeError("Protected study files changed; stop before inference")
    print("SCHEMA REPAIR PASSED: required streams, raw files, study locks and frozen rules preserved.", flush=True)
    print("Prepared queries:", manifest["query_count"], "checkpoints:", manifest["checkpoints"], flush=True)
    return manifest


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--base", type=Path, default=Path("/content/visual_memory"))
    parser.add_argument("--repo", type=Path, default=Path("/content/stretch_ai"))
    parser.add_argument("--scene", default="env2")
    args = parser.parse_args()
    stage_and_prepare(args.base, args.repo, args.scene)


if __name__ == "__main__":
    main()
PY_SCHEMA_ADAPTER
if [ -f "$FIX" ] && ! cmp -s "$FIX" "$FIX.incoming"; then
    echo "STOP: an existing repair script differs. It has not been replaced."
    rm -f "$FIX.incoming"
    exit 1
fi
mv "$FIX.incoming" "$FIX"

# Reuse downloaded env2. Stage an audited four-stream copy, then prepare it.
# This checks the frozen policies and original harness hashes before proceeding.
"$PY" "$FIX" --base "$BASE" --repo /content/stretch_ai --scene env2

# Only after the adapter and all existing validations pass, run the unchanged baseline.
"$PY" -m vmresearch.cli batch --base "$BASE" --scene env2
"$PY" -m vmresearch.cli score --base "$BASE" --scene env2


## 10. Compare rules and generate the report
Produces two distinct tables: **frozen-threshold performance** (coverage may shift on env2), and an
**exactly matched retrospective answer-quota comparison** using ranking scores, not answer correctness.
The matched table uses the same known-label cohort and tie-breaking rule for both methods; it is not an online threshold policy.

The baseline and always-decline comparator are included. With no answers, conditional error is **undefined**, not zero.
The report describes a win, tie, or loss only when measured. One test scene does not support a strong generalization claim.

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory

"$PY" -m vmresearch.cli evaluate --base "$BASE" > "$BASE/analysis/evaluation_console.json"
"$PY" -m vmresearch.cli report --base "$BASE"

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
report = Path("/content/visual_memory/report.md")
display(Markdown(report.read_text()))

## 11. Export the result bundle
Contains predictions, detector traces, evidence images, per-query scores, the convention audit, frozen rules,
comparison tables, provenance, and the small report. Full raw scenes, large memory arrays, and model weights are excluded.
Evidence images are still dataset-derived: verify permissions before publishing them.

Download the ZIP before the Colab session ends. **This is the file to return for review.**

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/content/DynaBench_Research_v3:/content/stretch_ai/src
PY=/content/stretch310/bin/python
BASE=/content/visual_memory
"$PY" -m vmresearch.cli export --base "$BASE"

In [ ]:
from pathlib import Path
from google.colab import files
archive = Path("/content/DynaBench_Research_v3_results.zip")
assert archive.is_file(), "Run the export cell first."
print("Results ZIP:", archive, f"({archive.stat().st_size:,} bytes)")
files.download(str(archive))

### Status and limits
- Implemented and locally tested: source, notebook structure/syntax, 57 synthetic/mock tests (43 research + 14 setup regression tests),
  actual env1 data adapter, and prior-result arithmetic.
- **Not run while authoring this notebook:** the new GPU detector audit, batches, or held-out rule comparison.
  Their output cells start empty; no successful execution is fabricated.
- **Still requires source evidence:** official checkpoint and blank-coordinate semantics.
  The notebook audits these and reports sensitivity; it does not certify them from data shape or scores.
- **May require human image review:** semantic identity of the selected toy. A box plus coordinate anchors
  is diagnostic evidence, not a certified object label. No evidence flags enter rule decisions from test labels.

The embedded README documents each module, resume behavior, interpretation, and pinned source links.

The setup regression tests mock installations/GPU availability. They do not certify a fresh GPU rebuild.
Setup audit files remain in `/content/visual_memory/setup_audits/` for troubleshooting.

**v3.2 repair validation:** 18 new schema-repair regression tests and the 57 existing tests passed locally. A real `env1` input-copy round trip also preserved all 42 frames and 54 queries. Tests include metadata-bearing synthetic scenes, tensor frames, input integrity, frozen-state preservation, safe resumption, and a mocked held-out workflow. The actual remote `env2` pickle and GPU held-out batch were not run here. The runtime cell validates its actual structure before proceeding.

**Post-run review (2026-10-27):** See `../docs/RESULTS.md` for independently checked results. At an equal quota of 18 scorable held-out answers, age-only had two geometric errors and age-plus-similarity had one. The difference is tie-sensitive, not a robust improvement. No official benchmark reproduction is claimed.
